In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interactive, FloatSlider, HBox, Layout, VBox, HTML
from IPython.display import display

def plot_bs_filter_specs(delta_p=0.08, delta_s=0.05, wp1=0.60, ws1=1.00, ws2=1.90, wp2=2.35):
    fig, ax = plt.subplots(figsize=(10, 6))
    omega = np.linspace(0.0, np.pi, 1200)

    H_mag = np.zeros_like(omega)

    for i, w in enumerate(omega):
        if w <= wp1:
            ratio = w / wp1 if wp1 > 0 else 0.0
            ripple = np.sin(4.0 * np.pi * ratio)
            H_mag[i] = 1.0 + delta_p * ripple

        elif wp1 < w < ws1:
            ratio = (w - wp1) / (ws1 - wp1)
            val_wp1 = 1.0 - delta_p
            val_ws1 = delta_s
            H_mag[i] = val_wp1 + ratio * (val_ws1 - val_wp1)

        elif ws1 <= w <= ws2:
            ratio = (w - ws1) / (ws2 - ws1)
            ripple = 0.5 * (1.0 + np.cos(5.0 * np.pi * ratio))
            H_mag[i] = delta_s * ripple

        elif ws2 < w < wp2:
            ratio = (w - ws2) / (wp2 - ws2)
            val_ws2 = delta_s
            val_wp2 = 1.0 - delta_p
            H_mag[i] = val_ws2 + ratio * (val_wp2 - val_ws2)

        else:
            ratio = (w - wp2) / (np.pi - wp2) if wp2 < np.pi else 0.0
            ripple = np.sin(4.0 * np.pi * ratio)
            H_mag[i] = 1.0 + delta_p * ripple

    upper_bound_pb = 1.0 + delta_p
    lower_bound_pb = 1.0 - delta_p
    stop_bound = delta_s

    ax.plot(omega, H_mag, 'r-', linewidth=2, label=r'$|H(e^{j\omega})|$')

    ax.hlines(upper_bound_pb, 0, wp1, colors='k', linewidth=1.5)
    ax.hlines(upper_bound_pb, wp2, np.pi, colors='k', linewidth=1.5)

    ax.hlines(lower_bound_pb, 0, wp1, colors='k', linewidth=1.5, linestyle='--')
    ax.hlines(lower_bound_pb, wp2, np.pi, colors='k', linewidth=1.5, linestyle='--')

    ax.hlines(stop_bound, ws1, ws2, colors='k', linewidth=1.5)

    ax.axvline(wp1, ymin=0, ymax=0.8, color='k', linestyle='--', linewidth=1)
    ax.axvline(ws1, ymin=0, ymax=0.35, color='k', linestyle='--', linewidth=1)
    ax.axvline(ws2, ymin=0, ymax=0.35, color='k', linestyle='--', linewidth=1)
    ax.axvline(wp2, ymin=0, ymax=0.8, color='k', linestyle='--', linewidth=1)

    ax.fill_between([0, wp1], upper_bound_pb, 1.25, color='blue', alpha=0.1, hatch='//')
    ax.fill_between([0, wp1], 0.0, lower_bound_pb, color='blue', alpha=0.1, hatch='//')

    ax.fill_between([ws1, ws2], stop_bound, 1.0, color='blue', alpha=0.1, hatch='//')

    ax.fill_between([wp2, np.pi], upper_bound_pb, 1.25, color='blue', alpha=0.1, hatch='//')
    ax.fill_between([wp2, np.pi], 0.0, lower_bound_pb, color='blue', alpha=0.1, hatch='//')

    ax.set_xlim(0, np.pi)
    ax.set_ylim(-0.02, 1.25)

    ax.set_xlabel(r'$\omega$', fontsize=14)
    ax.set_ylabel(r'$|H(e^{j\omega})|$', fontsize=14)

    ax.set_xticks([wp1, ws1, ws2, wp2, np.pi])
    ax.set_xticklabels([r'$\omega_{p1}$', r'$\omega_{s1}$', r'$\omega_{s2}$', r'$\omega_{p2}$', r'$\pi$'], fontsize=12)

    y_ticks = [stop_bound, lower_bound_pb, 1.0, upper_bound_pb]
    y_labels = [r'$\delta_s$', r'$1-\delta_p$', '1', r'$1+\delta_p$']

    ax.set_yticks(y_ticks)
    ax.set_yticklabels(y_labels, fontsize=12)

    ax.text(wp1 / 2, 0.5, 'Passband 1', color='green', fontsize=10, fontweight='bold', ha='center')

    ax.text((wp1 + ws1) / 2, 0.5, 'Transition\nBand', color='green', fontsize=9, fontweight='bold', ha='center')

    ax.text((ws1 + ws2) / 2, 0.5, 'Stopband', color='green', fontsize=11, fontweight='bold', ha='center')

    ax.text((ws2 + wp2) / 2, 0.5, 'Transition\nBand', color='green', fontsize=9, fontweight='bold', ha='center')

    ax.text(wp2 + (np.pi - wp2) / 2, 0.5, 'Passband 2', color='green', fontsize=10, fontweight='bold', ha='center')

    ax.grid(True, linestyle=':', alpha=0.6)

    plt.title('Normalized Frequency Response of a Band-Stop Digital Filter', fontsize=13, pad=15)

    plt.show()


slider_layout = Layout(width='310px')

style_opts = {'description_width': '55px'}

dp_slider = FloatSlider(min=0.01, max=0.20, step=0.01, value=0.08, description='δp:', style=style_opts, layout=slider_layout)

ds_slider = FloatSlider(min=0.01, max=0.20, step=0.01, value=0.05, description='δs:', style=style_opts, layout=slider_layout)

wp1_slider = FloatSlider(min=0.20, max=0.80, step=0.05, value=0.60, description='ωp1:', style=style_opts, layout=slider_layout)

ws1_slider = FloatSlider(min=0.85, max=1.30, step=0.05, value=1.00, description='ωs1:', style=style_opts, layout=slider_layout)

ws2_slider = FloatSlider(min=1.50, max=2.10, step=0.05, value=1.90, description='ωs2:', style=style_opts, layout=slider_layout)

wp2_slider = FloatSlider(min=2.15, max=3.00, step=0.05, value=2.35, description='ωp2:', style=style_opts, layout=slider_layout)

widget_plot = interactive(plot_bs_filter_specs, delta_p=dp_slider, delta_s=ds_slider, wp1=wp1_slider, ws1=ws1_slider, ws2=ws2_slider, wp2=wp2_slider)

theory_html = HTML("""
<div style="font-family: monospace; font-size: 13px; line-height: 1.5; margin-bottom: 8px;">
<b>δp:</b> Maximum allowed passband deviation.<br>
<b>δs:</b> Maximum allowed stopband magnitude.<br>
<b>ωp1, ωp2:</b> Passband edge frequencies.<br>
<b>ωs1, ωs2:</b> Stopband edge frequencies.
</div>
""")

controls = VBox([dp_slider, ds_slider, wp1_slider, ws1_slider, ws2_slider, wp2_slider], layout=Layout(width='340px', min_width='340px', overflow='visible', justify_content='center'))

main_layout = HBox([widget_plot.children[-1], controls], layout=Layout(width='100%', overflow='visible', align_items='center', justify_content='flex-start'))

display(VBox([theory_html, main_layout], layout=Layout(width='100%', overflow='visible', align_items='flex-start')))